# **MedPub Rag Answering Chatbot**

## 0. Libraries

In [ ]:
!pip install -q biopython langchain langchain-community langchain-text-splitters langchain-openai faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.4/120.4 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
!pip show langchain

Name: langchain
Version: 1.3.11
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 


In [ ]:
import os
from dotenv import load_dotenv
from datetime import datetime
import time
from typing import List, Dict, Any
import json
from google.colab import userdata

from Bio import Entrez, Medline
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_core.load import dumps, loads

import pandas as pd

/tmp/ipykernel_3246/2399962503.py:14: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [ ]:
# Loading environment variables
openai_api_key = userdata.get("OPENAI_API_KEY")
if not openai_api_key :
  print("The OPENAI_API_KEY could not be found in Colab Secrets. Please add it to your Secrets.")


# # 환경 변수 로딩
# load_dotenv()
# openai_api_key = os.getenv("OPENAI_API_KEY", "")
# if not openai_api_key:
#    print(".env 파일에서 OPENAI_API_KEY를 찾지 못했습니다. 환경변수를 설정해주세요.")

## 1. PubMed Search Function

In [ ]:
Entrez.email = 'jeyuna220@gmail.com'
Entrez.tool = "PubMedEducation"

In [ ]:
# PubMed search function
def search_pubmed(query: str, email: str, max_results: int, start_year: int, end_year: int) -> List[Dict]:

    Entrez.email = email

    # Build search query with date filter
    date_filter = f" AND {start_year}:{end_year}[dp]"
    full_query = query + date_filter

    try:
        # Perform search
        handle = Entrez.esearch(
            db="pubmed",
            term=full_query,
            retmax=max_results,
            sort="relevance"
        )
        search_results = Entrez.read(handle)
        handle.close()

        id_list = search_results["IdList"]

        if not id_list:
            return []

        # Fetch detailed paper information
        handle = Entrez.efetch(
            db="pubmed",
            id=id_list,
            rettype="medline",
            retmode="text")

        papers = []
        records = list(Medline.parse(handle))
        handle.close()

        for record in records:
            paper = {
                "pmid": record.get("PMID", ""),
                "title": record.get("TI", "No title"),
                "abstract": record.get("AB", "No abstract available"),
                "authors": ", ".join(record.get("AU", ["No author information"])),
                "journal": record.get("TA", "No journal information"),
                "year": record.get("DP", "No year available").split()[0] if record.get("DP") else "No year available",
                "doi": next((ref for ref in record.get("AID", []) if ref.endswith("[doi]")), "").replace("[doi]", ""),
                "types": record.get("PT", []),
            }

            # Build full text
            # If abstract is missing, try to fetch it separately
            if not paper["abstract"] or paper["abstract"].strip() in {"", "No abstract available"}:
                try:
                    ah = Entrez.efetch(
                        db="pubmed",
                        id=paper["pmid"],
                        rettype="abstract",
                        retmode="text")

                    abstract_txt = ah.read()
                    ah.close()

                    if isinstance(abstract_txt, bytes):
                        abstract_txt = abstract_txt.decode("utf-8", errors="ignore")

                    if abstract_txt and abstract_txt.strip():
                        paper["abstract"] = abstract_txt.strip()

                except Exception:
                    pass

            paper["full_text"] = f"""
                Title: {paper['title']}
                Authors: {paper['authors']}
                Journal: {paper['journal']}
                Year: {paper['year']}
                PMID: {paper['pmid']}
                DOI: {paper['doi']}
                Abstract:{paper['abstract']}
            """

            papers.append(paper)

        return papers

    except Exception as e:
        print(f"Error occurred during PubMed search: {str(e)}")
        return []

## 2. Indexing

In [ ]:
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# Chunking
def chunk_papers(papers):
  splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 50)

  chunks = []
  for paper in papers :
    texts = splitter.split_text(paper["full_text"])
    for t in texts :
      chunks.append({
          "text"   : t,
          "pmid"   : paper['pmid'],
          "title"  : paper['title'],
          "authors": paper["authors"],
          "journal": paper["journal"],
          "year": paper["year"],
          "abstract": paper["abstract"]
      })

  return chunks

In [ ]:
# Indexing
def build_faiss(papers):
    chunks = chunk_papers(papers)

    vectorstore = FAISS.from_texts(
        texts=[c["text"] for c in chunks],
        embedding=OpenAIEmbeddings(api_key=os.environ["OPENAI_API_KEY"]),
        metadatas=[{
            "pmid"   : c["pmid"],
            "title"  : c["title"],
            "authors" : c["authors"],
            "journal" : c["journal"],
            "year"    : c["year"],
            "abstract": c["abstract"]
        } for c in chunks]
    )
    return vectorstore

## 2. Query Optimisiaion

### 2-1. Muti-query
Generating multiple versions of a single question to retrieve more comprehensive results form a vector database

In [ ]:
# Muti-query
def get_unique_union(documents: list[list]):
    flattened_docs = [dumps(doc) for sublist in documents for doc in sublist]
    unique_docs = list(set(flattened_docs))
    return [loads(doc) for doc in unique_docs]


def query_multiquery(question, retriever):
    # 1. template
    template = """You are an AI language model assistant. Your task is to generate five
    different versions of the given user question to retrieve relevant documents from a vector
    database. Provide these alternative questions separated by newlines.
    Original question: {question}"""
    prompt_perspectives = ChatPromptTemplate.from_template(template)

    # 2. query generation
    generate_queries = (
        prompt_perspectives
        | ChatOpenAI(temperature=0)
        | StrOutputParser()
        | (lambda x: x.split("\n"))
    )

    # 3. retrieval chain + execute
    retrieval_chain = generate_queries | retriever.map() | get_unique_union
    docs = retrieval_chain.invoke({"question": question})
    return docs

### 2-2. Step-back
Instead of answering the question directly, first ask a broader, more abstract version of the question to better background knowledge.

In [ ]:
def query_stepback(question, retriever):
    # 1. Step-back example
    examples = [
        {
            "input": "Could the members of The Police perform lawful arrests?",
            "output": "What can the members of The Police do?",
        },
        {
            "input": "Jan Sindel's was born in what country?",
            "output": "What is Jan Sindel's personal history?",
        },
    ]

    example_prompt = ChatPromptTemplate.from_messages([
        ("human", "{input}"),
        ("ai", "{output}"),
    ])

    few_shot_prompt = FewShotChatMessagePromptTemplate(
        example_prompt=example_prompt,
        examples=examples
    )

    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are an expert at world knowledge. Your task is to step back
        and paraphrase a question to a more generic step-back question, which is
        easier to answer. Here are a few examples:"""),
        few_shot_prompt,
        ("user", "{question}"),
    ])

    generate_queries_step_back = prompt | ChatOpenAI(temperature=0) | StrOutputParser()

    # 2. step-back question generation
    step_back_question = generate_queries_step_back.invoke({"question": question})

    # 3. Retrieve using both original and step-back question
    normal_docs = retriever.invoke(question)
    step_back_docs = retriever.invoke(step_back_question)

    # 4. Merge results
    #docs = normal_docs + step_back_docs
    docs = get_unique_union([normal_docs, step_back_docs])
    return docs

### 2-3. HyDE

Instead of searching with the question, generate a hypotheircal anwer first, then search with that anwser

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI


def query_hyde(question, retriever):
    # 1. Prompt template for generating a hypothetical document
    template = """Please write a scientific paper passage to answer the question.
    Question: {question}
    Passage:"""
    prompt_hyde = ChatPromptTemplate.from_template(template)

    generate_docs_for_retrieval = (
        prompt_hyde
        | ChatOpenAI(temperature=0)
        | StrOutputParser()
    )

    # 2. Generate a hypothetical document (keep the original question unchanged)
    hypothetical_doc = generate_docs_for_retrieval.invoke({"question": question})

    # 3. Retrieve relevant document using hypothetical documnet
    docs = retriever.invoke(hypothetical_doc)

    return docs


## 3. Generation

## 3-1. Select query

In [ ]:
# Optimise question and select the documents
def run_query(question, method, retriever):
    """
    method: "original" | "multiquery" | "hyde" | "stepback"
    """
    if method == "original":
        return retriever.invoke(question)

    if method == "multiquery":
        return query_multiquery(question, retriever)

    if method == "hyde":
        return query_hyde(question, retriever)

    if method == "stepback":
        return query_stepback(question, retriever)

    raise ValueError(f"Unknown method: {method}")


## 3-2. Generate answers

In [ ]:
def generate_answer(question, docs):
    context = "\n\n".join([d.page_content for d in docs])

    template = """
You are a medical AI assistant that provides accurate information based on published medical research articles.
Answer the user's question using only the information provided in the supplied papers. Do not use outside knowledge or make assumptions.
Always cite the source for every relevant statement, including the PMID, author(s), and publication year.
If the provided papers do not contain enough information to answer the question, clearly state that the evidence is insufficient rather than speculating.

Context:
{context}

Question:
{question}
"""
    prompt = ChatPromptTemplate.from_template(template)
    llm = ChatOpenAI(temperature=0)
    chain = prompt | llm | StrOutputParser()

    answer =  chain.invoke({"context": context, "question": question})

    # Get the whole information while removing duplicates
    seen = set()
    sources = []
    for d in docs:
        pmid = d.metadata.get("pmid", "")
        if pmid not in seen:
            seen.add(pmid)
            sources.append({
                "title": d.metadata.get("title", ""),
                "authors": d.metadata.get("authors", ""),
                "journal": d.metadata.get("journal", ""),
                "year": d.metadata.get("year", ""),
                "pmid": pmid,
                "abstract": d.metadata.get("abstract", "")
            })

    return answer, sources


In [ ]:
# PubMed search
papers = search_pubmed(
    query="metformin type 2 diabetes",
    email=Entrez.email,
    max_results=10,
    start_year=2020,
    end_year=2024
)
print(f"Paper {len(papers)} searched")

# FAISS indexing
vs = build_faiss(papers)
print("FAISS Indexing done")

# Retriever
retriever = vs.as_retriever(search_kwargs={"k": 5})

Paper 10 searched
FAISS Indexing done


In [ ]:
# test

# config
config = {"query": "hyde"}    # "original" | "multiquery" | "hyde" | "stepback"
question = "What is metformin?"
docs = run_query(question, config["query"], retriever)
answer, sources = generate_answer(question, docs)

# Print anwser
print(answer)
print("---------------")
for s in sources:
    print(f"- Title: {s['title']}")
    print(f"  Authors: {s['authors']}")
    print(f"  Journal: {s['journal']} ({s['year']})")
    print(f"  PMID: {s['pmid']}")

Metformin is a first-line therapy for the treatment of type 2 diabetes mellitus. It is an antihyperglycemic agent that primarily works by activating AMPK (Adenosine Monophosphate-Activated Protein Kinase) in cells and reducing glucose output from the liver. Metformin reduces glucose levels by decreasing hepatic glucose production, reducing intestinal glucose absorption, and increasing insulin sensitivity. It has been in practical use for the last six decades and continues to be the preferred drug for newly diagnosed type 2 diabetes mellitus (PMID: 32062637, 32103414, 32804777, 33323707).
---------------
- Title: Cellular and Molecular Mechanisms of Metformin Action.
  Authors: LaMoia TE, Shulman GI
  Journal: Endocr Rev (2021)
  PMID: 32897388
- Title: Metformin: A Review of Potential Mechanism and Therapeutic Utility Beyond Diabetes.
  Authors: Dutta S, Shah RB, Singhal S, Dutta SB, Bansal S, Sinha S, Haque M
  Journal: Drug Des Devel Ther (2023)
  PMID: 37397787
- Title: Metformin: P

In [ ]:
#
# HyDE는 가상의 "과학 논문 구절"을 만들어서 검색했으니까, 더 전문적이고 메커니즘 중심의 chunks를 찾아온 것 같아 (AMPK 같은 분자생물학 용어).
# Multi-Query는 5가지 다른 관점의 질문을 던졌으니까, 더 다양한 각도(개요, 사용법, 병용요법)의 chunks를 가져온 것 같고.

## 4. Retreival

- Rettrieve (search)
- Grade_documents (relevant evaluation)
- decide to generate (분기)
- generate

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

# 1. Retrieval Grader
class GradeDocuments(BaseModel):

    # Pydantic grammer
    binary_score: str = Field(description="Documents are relevant to the question, 'yes' or 'no'")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# force the form
structured_llm_grader = llm.with_structured_output(GradeDocuments)

system = """You are a grader assessing relevance of a retrieved document to a user question.
    If the document contains keyword(s) or semantic meaning related to the question, grade it as relevant.
    Give a binary score 'yes' or 'no' score to indicate whether the document is relevant to the question."""

grade_prompt = ChatPromptTemplate.from_messages([
    ("system", system),
    ("human", "Retrieved document: \n\n {document} \n\n User question: {question}"),
])

retrieval_grader = grade_prompt | structured_llm_grader

In [ ]:
# 2. Question Re-writer
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

system = """You are a question re-writer that converts an input question to a better version
that is optimized for searching PubMed medical literature.
Look at the input and try to reason about the underlying medical/scientific intent."""

re_write_prompt = ChatPromptTemplate.from_messages([
    ("system", system),
    ("human", "Here is the initial question: \n\n {question} \n Formulate an improved question."),
])

question_rewriter = re_write_prompt | llm | StrOutputParser()


In [ ]:
# 3. GraphState
# A shared memory object that all nodes in LangGraph can read from and write to
from typing import TypedDict, List
class GraphState(TypedDict):
  question : str              # user's original question
  generation : str            # LLM generated anwser
  web_search : str            # whether web sestvh id needed
  documents : List[str]       # retrieved documnets
  query_method : str

In [ ]:
# 4. node function

def retrieve(state):
    print("---RETRIEVE---")
    question = state["question"]
    method = state["query_method"]

    documents = run_query(question, method, retriever)
    return {"documents": documents, "question": question}


def generate(state):
    print("---GENERATE---")
    question = state["question"]
    documents = state["documents"]
    answer, sources = generate_answer(question, documents)
    return {"documents": documents, "question": question, "generation": answer}


def grade_documents(state):
    print("---CHECK DOCUMENT RELEVANCE---")
    question = state["question"]
    documents = state["documents"]

    filtered_docs = []
    web_search = "No"
    for d in documents:
        score = retrieval_grader.invoke({"question": question, "document": d.page_content})
        if score.binary_score == "yes":
            print("---GRADE: RELEVANT---")
            filtered_docs.append(d)
        else:
            print("---GRADE: NOT RELEVANT---")
            web_search = "Yes"

    return {"documents": filtered_docs, "question": question, "web_search": web_search}


def transform_query(state):
    print("---TRANSFORM QUERY---")
    question = state["question"]
    documents = state["documents"]
    better_question = question_rewriter.invoke({"question": question})
    return {"documents": documents, "question": better_question}


def research_again(state):
    print("---PUBMED RE-SEARCH---")
    question = state["question"]
    documents = state["documents"]

    new_papers = search_pubmed(
        query=question,
        email=Entrez.email,
        max_results=10,
        start_year=2020,
        end_year=2024
    )
    if new_papers:
        new_vs = build_faiss(new_papers)
        new_retriever = new_vs.as_retriever(search_kwargs={"k": 5})
        new_docs = new_retriever.invoke(question)
        documents.extend(new_docs)

    return {"documents": documents, "question": question}


def decide_to_generate(state):
    print("---ASSESS GRADED DOCUMENTS---")
    web_search = state["web_search"]

    if web_search == "Yes":
        print("---DECISION: NOT RELEVANT, TRANSFORM QUERY---")
        return "transform_query"
    else:
        print("---DECISION: GENERATE---")
        return "generate"


In [ ]:
# 5. Connect graph
from langgraph.graph import StateGraph, START, END
workflow = StateGraph(GraphState)

# Add node
workflow.add_node("retrieve", retrieve)
workflow.add_node("grade_documents", grade_documents)
workflow.add_node("generate", generate)
workflow.add_node("transform_query", transform_query)
workflow.add_node("research_again", research_again)

# connect edge
workflow.add_edge(START, "retrieve")
workflow.add_edge("retrieve", "grade_documents")
workflow.add_conditional_edges(
    "grade_documents",
    decide_to_generate,
    {
        "transform_query": "transform_query",
        "generate": "generate"
    }
)
workflow.add_edge("transform_query", "research_again")
workflow.add_edge("research_again", "generate")
workflow.add_edge("generate", END)

app = workflow.compile()

In [ ]:
# input

inputs = {
    "question" : "What is metformin?",
    "query_method" : "multiquery"
}

for output in app.stream(inputs) :
  for key, value in output.items():
    print(f"Node '{key}' : ")

print(value['generation'])


---RETRIEVE---


/tmp/ipykernel_3246/4050141576.py:5: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  return [loads(doc) for doc in unique_docs]
/tmp/ipykernel_3246/4050141576.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(doc) for doc in unique_docs]


Node 'retrieve' : 
---CHECK DOCUMENT RELEVANCE---
---GRADE: RELEVANT---
---GRADE: RELEVANT---
---GRADE: RELEVANT---
---GRADE: RELEVANT---
---GRADE: RELEVANT---
---GRADE: RELEVANT---
---GRADE: RELEVANT---
---GRADE: RELEVANT---
---GRADE: RELEVANT---
---GRADE: NOT RELEVANT---
---GRADE: RELEVANT---
---GRADE: RELEVANT---
---GRADE: RELEVANT---
---GRADE: RELEVANT---
---ASSESS GRADED DOCUMENTS---
---DECISION: NOT RELEVANT, TRANSFORM QUERY---
Node 'grade_documents' : 
---TRANSFORM QUERY---
Node 'transform_query' : 
---PUBMED RE-SEARCH---
Node 'research_again' : 
---GENERATE---
Node 'generate' : 
The pharmacological properties of metformin include its ability to reduce glucose output from the liver, activate AMPK in cells, decrease advanced glycation end products, and have pleiotropic effects on various systems and processes (Triggle et al., 2022). The major glucose-lowering effect of metformin in patients with type 2 diabetes is mostly mediated through the inhibition of hepatic gluconeogenesis 

In [ ]:
# Stage 1: making dictionary
keywords = ["metformin", "diabetes type 2", "metformin mechanism",
            "metformin side effects", "metformin cancer"]

all_papers = []
for kw in keywords:
    papers = search_pubmed(kw, email=EMAIL, max_results=20, start_year=2020, end_year=2024)
    all_papers.extend(papers)

print(f" Collected {len(all_papers)} papers")

vs = build_faiss(all_papers)
vs.save_local("pubmed_faiss_index")  # save the index for the future

# Stage 2: Using
vs = FAISS.load_local("pubmed_faiss_index", OpenAIEmbeddings(),
                       allow_dangerous_deserialization=True)
retriever = vs.as_retriever(search_kwargs={"k": 5})

# Ask questions directly without querying PubMed again.
question = "What is metformin?"
docs = retriever.invoke(question)